# Module 03 — Prompt & Context Engineering

Same model, two very different results — the difference is the *prompt*. This module is the **craft** of getting reliable answers:

- **Prompt engineering** — how you word instructions, show examples, and demand a format.
- **Context engineering** — *what* information you put in the (finite) context window.

Most "the AI got it wrong" moments are really prompt problems. Run each cell top to bottom and watch the quality change.

## 0. Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model
# temperature=0 -> repeatable answers, so we can compare prompts fairly.
model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


## 1. Specificity — vague in, vague out

A model can't read your mind; it fills gaps with *its* assumptions. A vague prompt gets a generic answer. A specific one — audience, length, format, constraints — gets what you actually wanted.

Run both and compare. Same topic, wildly different usefulness.

In [2]:
vague = "Tell me about Python."

specific = (
    "Explain what the Python programming language is to a complete beginner. "
    "Use exactly 3 bullet points, each one sentence, no jargon."
)

print("VAGUE:", model.invoke(vague).text)
print("" + "=" * 60 + "")
print("SPECIFIC:", model.invoke(specific).text)

VAGUE: Python is one of the most popular and widely used programming languages in the world. It's known for its simplicity, readability, and versatility, making it a favorite among beginners and experienced developers alike.

Here's a breakdown of what makes Python so special:

---

### What is Python?

*   **High-Level:** Python abstracts away many low-level details (like memory management), allowing developers to focus on solving problems rather than managing system resources.
*   **Interpreted:** Python code is executed line by line by an interpreter, rather than being compiled into machine code beforehand. This makes the development cycle faster, as you can run code immediately after writing it.
*   **General-Purpose:** It's not designed for a specific task but can be used for a wide variety of applications across different domains.
*   **Object-Oriented:** Python supports object-oriented programming (OOP) paradigms, allowing for modular and reusable code.
*   **Dynamically Typed:*

## 2. Few-shot prompting — show, don't just tell

Sometimes describing the format is hard, but *showing* it is easy. **Few-shot** prompting means putting a few example input→output pairs in the prompt. The model copies the pattern.

We model each example as a `human` turn (the input) followed by an `ai` turn (the desired output). Here we teach a made-up format — turn a word into an emoji summary — with two examples, then give a new word.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

few_shot = ChatPromptTemplate.from_messages([
    ("system", "You turn a word into exactly three emojis that represent it. Output only the emojis."),
    ("human", "pizza"),
    ("ai", "🍕🧀🍅"),
    ("human", "rain"),
    ("ai", "🌧️☔💧"),
    ("human", "{word}"),   # the new input
])

prompt = few_shot.invoke({"word": "birthday"})
print(model.invoke(prompt).text)

🎂🥳🎁


## 3. Delimiters & explicit output format

When your prompt mixes *instructions* and *user data*, the model can confuse the two. Fence the data with **delimiters** (triple backticks, XML-like tags) so it's unmistakable. Then state the **exact output format** you want — especially if code will parse it.

Below: the text to analyze is fenced, and we demand a strict one-line format.

In [4]:
user_text = "I love the battery life but the camera is disappointing and the price is too high."

prompt = f'''Analyze the customer review delimited by triple backticks.

Output EXACTLY this format, nothing else:
SENTIMENT: <positive|negative|mixed>
TOPICS: <comma-separated list>

```
{user_text}
```'''

print(model.invoke(prompt).text)

SENTIMENT: mixed
TOPICS: Battery Life, Camera, Price


## 4. Step-by-step reasoning

For anything with logic — math, multi-step decisions, puzzles — telling the model to **think step by step before answering** raises accuracy a lot. Jumping straight to an answer is where models slip.

Run the two versions of the same word problem and compare the reasoning (and whether the final number is right).

In [5]:
problem = (
    "A shop sells pens at 3 for $2. I have $10. "
    "How many pens can I buy, and how much change do I get?"
)

rushed = problem + " Answer with just the final numbers."
stepwise = problem + " Think step by step, show your work, then give the final answer."

print("RUSHED:", model.invoke(rushed).text)
print("" + "=" * 60 + "")
print("STEP BY STEP:", model.invoke(stepwise).text)

RUSHED: 15 0
STEP BY STEP: Here's how to solve this step-by-step:

**Step 1: Determine how many sets of pens you can buy.**
The pens are sold in sets of 3 for $2. You have $10.
Number of sets = Total money / Cost per set
Number of sets = $10 / $2 = 5 sets

**Step 2: Calculate the total number of pens you can buy.**
Each set contains 3 pens. You can buy 5 sets.
Total pens = Number of sets * Pens per set
Total pens = 5 * 3 pens = 15 pens

**Step 3: Calculate the total cost.**
You bought 5 sets, and each set costs $2.
Total cost = Number of sets * Cost per set
Total cost = 5 * $2 = $10

**Step 4: Calculate the change.**
You started with $10 and spent $10.
Change = Starting money - Total cost
Change = $10 - $10 = $0

---

**Final Answer:**
You can buy **15 pens**, and you will get **$0** in change.


## 5. The context window — and what to put in it

Everything you send — system message, examples, fenced data, conversation history — shares **one finite budget** called the **context window** (measured in tokens, Module 00). When it's full, something must be cut.

So a real skill is **context engineering**: include what's *relevant*, leave out what isn't. Watch how giving the model the right context changes a question it otherwise can't answer.

In [6]:
question = "What is our refund window, and does it apply to sale items?"

# Without context: the model guesses or hedges.
print("NO CONTEXT:", model.invoke(question).text)

# With the relevant policy in context: it answers from YOUR facts.
context = '''Company policy:
- Refunds accepted within 30 days of purchase.
- Sale and clearance items are final — no refunds.
- Refunds are issued to the original payment method.'''

grounded = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer ONLY using the policy below. If it's not covered, say you don't know.
        {policy}"""
    ),
    ("human", "{question}"),
])
prompt = grounded.invoke({"policy": context, "question": question})
print("" + "=" * 60 + "")
print("WITH CONTEXT:", model.invoke(prompt).text)

NO CONTEXT: As an AI, I don't have a "we" or a specific company's refund policy. My purpose is to provide information.

To find out the refund window and policy for sale items for a specific retailer, you'll need to check their official policy. Here's where you typically find it and what to look for:

**Where to Find a Retailer's Refund Policy:**

1.  **Website Footer:** Look for links like "Returns," "Refunds," "Shipping & Returns," "Customer Service," or "FAQ" at the very bottom of their website.
2.  **FAQ Page:** Many common questions, including returns, are answered here.
3.  **Product Page:** Sometimes, specific return conditions (especially for final sale items) are listed directly on the product's description page.
4.  **Terms and Conditions:** The full legal policy will be here, though it can be dense.
5.  **Receipt or Order Confirmation:** Your physical or email receipt often has a summary of the return policy.
6.  **Contact Customer Service:** If you can't find it, reach out 

## Recap

- **Specificity** — state audience, length, format, constraints; don't make the model guess.
- **Few-shot** — show example input→output pairs and the model copies the pattern.
- **Delimiters + exact format** — fence user data, demand a precise output shape so it's parseable.
- **Step-by-step** — ask the model to reason before answering on anything with logic.
- **Context window** — one finite token budget; **context engineering** = include only what's relevant. This is the manual version of what **RAG** automates later.

## 🛠️ Try it yourself

1. Take a vague prompt of your own and rewrite it to be specific. Run both, compare.
2. Build a few-shot template for a format *you* invent (e.g. turn a sentence into a hashtag). Two examples, then a new input.
3. Break section 3's output format on purpose (ask for the wrong shape) and watch parsing get harder — that's why the spec matters.
4. Find a logic puzzle where "step by step" gets it right but "just the answer" gets it wrong.
5. **Stretch:** in section 5, ask something the policy *doesn't* cover and confirm the model says "I don't know" instead of inventing.

When you're done, say **"next"** and we'll build **Module 04 — Structured Output**.